# 북항, 신항 환적 효율성

- `data.csv`가 북항/신항이 환적항으로 얼마나 효율적으로 운영되는가 라는 질문에 답할 수 있는 데이터인가?


In [6]:
import pandas as pd

TARGET_PATH = 'data/eda_data.csv'

In [7]:
df = pd.read_csv(TARGET_PATH, encoding='euc-kr')

In [ ]:
df.shape
# 2409값 17 컬럼 

(2409, 17)

In [11]:
df.head(3)

,연도,월,분기,청코드,내외항구분,수출입구분명,시설코드,시설명,부두구분명,아외국구분,적공구분,컨테이너수(10피트),컨테이너수(20피트),컨테이너수(40피트),컨테이너수(기타),전체개수,전체물동량
0,2024,3,1,신항,외항,수입,6,신항 W 정박지,일반부두,외국선,적컨,0,3,8,0,11,19.0
1,2024,4,2,신항,외항,수입,6,신항 W 정박지,일반부두,아국선,적컨,0,9,3,0,12,15.0
2,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,공컨,0,1,0,0,1,1.0


In [21]:
df['전체물동량'].isna().sum()

np.int64(0)

In [12]:
df.columns.to_list()

['연도',
 '월',
 '분기',
 '청코드',
 '내외항구분',
 '수출입구분명',
 '시설코드',
 '시설명',
 '부두구분명',
 '아외국구분',
 '적공구분',
 '컨테이너수(10피트)',
 '컨테이너수(20피트)',
 '컨테이너수(40피트)',
 '컨테이너수(기타)',
 '전체개수',
 '전체물동량']

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2409 entries, 0 to 2408
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   연도           2409 non-null   int64  
 1   월            2409 non-null   int64  
 2   분기           2409 non-null   int64  
 3   청코드          2409 non-null   str    
 4   내외항구분        2409 non-null   str    
 5   수출입구분명       2409 non-null   str    
 6   시설코드         2409 non-null   int64  
 7   시설명          2409 non-null   str    
 8   부두구분명        2409 non-null   str    
 9   아외국구분        2409 non-null   str    
 10  적공구분         2409 non-null   str    
 11  컨테이너수(10피트)  2409 non-null   int64  
 12  컨테이너수(20피트)  2409 non-null   int64  
 13  컨테이너수(40피트)  2409 non-null   int64  
 14  컨테이너수(기타)    2409 non-null   int64  
 15  전체개수         2409 non-null   int64  
 16  전체물동량        2409 non-null   float64
dtypes: float64(1), int64(9), str(7)
memory usage: 320.1 KB


In [9]:
for col in ['청코드', '내외항구분', '수출입구분명', '아외국구분','적공구분','부두구분명']:
    print(col, ":", df[col].unique())
    print()

청코드 : <ArrowStringArray>
['신항', '북항', '감천']
Length: 3, dtype: str

내외항구분 : <ArrowStringArray>
['외항']
Length: 1, dtype: str

수출입구분명 : <ArrowStringArray>
['수입', '수입환적', '수출환적', '수출']
Length: 4, dtype: str

아외국구분 : <ArrowStringArray>
['외국선', '아국선']
Length: 2, dtype: str

적공구분 : <ArrowStringArray>
['적컨', '공컨']
Length: 2, dtype: str

부두구분명 : <ArrowStringArray>
['일반부두', '컨테이너부두']
Length: 2, dtype: str



In [19]:
df.groupby(['청코드', '월']).size().unstack(level=0)

청코드,감천,북항,신항
월,,,
1,7,87,97
2,9,87,101
3,11,94,98
4,8,77,107
5,13,75,113
6,9,70,110
7,10,79,116
8,8,70,112
9,7,89,114


## 목표
- 환적을 얼마나 많이 하는가?(규모)
- 환적을 얼마나 효율적으로 처리하는가?(효율)

In [15]:
# 환적 비중 계산
df['환적여부'] = df['수출입구분명'].isin(['수입환적', '수출환적'])

In [16]:
zone_total = df.groupby('청코드')['전체물동량'].sum()
zone_total

청코드
감천        9014.0
북항     6512510.0
신항    17880496.0
Name: 전체물동량, dtype: float64

In [17]:
zone_transship = df[df['환적여부']].groupby('청코드')['전체물동량'].sum()
zone_transship

청코드
감천        6192.00
북항     2314512.75
신항    11176479.25
Name: 전체물동량, dtype: float64

In [18]:
zone_share = (zone_transship / zone_total * 100).round(2)

zone_share

청코드
감천    68.69
북항    35.54
신항    62.51
Name: 전체물동량, dtype: float64

In [21]:
df.columns

Index(['연도', '월', '분기', '청코드', '내외항구분', '수출입구분명', '시설코드', '시설명', '부두구분명',
       '아외국구분', '적공구분', '컨테이너수(10피트)', '컨테이너수(20피트)', '컨테이너수(40피트)',
       '컨테이너수(기타)', '전체개수', '전체물동량', '환적여부'],
      dtype='str')

In [19]:
teu_estimate = df['컨테이너수(10피트)'] * 0.5 + df['컨테이너수(20피트)'] * 1 + df['컨테이너수(40피트)'] * 2

In [42]:
teu_estimate.corr(df['전체물동량'])

np.float64(0.9999898787140992)

In [23]:
grp = df.groupby(['청코드', '환적여부']).agg(물동량=('전체물동량', 'sum'), 개수=('전체개수', 'sum'))
grp['TEU_FACTOR'] = grp['물동량'] / grp['개수']

grp

물동량       개수  TEU_FACTOR
청코드 환적여부                                   
감천  False      2822.00     1516    1.861478
    True       6192.00     3210    1.928972
북항  False   4197997.25  2859718    1.467976
    True    2314512.75  1569611    1.474577
신항  False   6704016.75  4140278    1.619219
    True   11176479.25  6464959    1.728778

In [10]:
df.columns

Index(['연도', '월', '분기', '청코드', '내외항구분', '수출입구분명', '시설코드', '시설명', '부두구분명',
       '아외국구분', '적공구분', '컨테이너수(10피트)', '컨테이너수(20피트)', '컨테이너수(40피트)',
       '컨테이너수(기타)', '전체개수', '전체물동량'],
      dtype='str')

In [22]:
df.groupby(['청코드','적공구분','환적여부'])['전체물동량'].sum().unstack(fill_value = 0)

환적여부           False        True 
청코드 적공구분                         
감천  공컨       1083.00         0.00
    적컨       1739.00      6192.00
북항  공컨    1314340.50     45790.00
    적컨    2883656.75   2268722.75
신항  공컨    1983822.00    713056.00
    적컨    4720194.75  10463423.25

In [ ]:
grp2(['공컨'] / grp(['공컨'] + grp2(['적컨']) * 100 )) 
# 비율 계산 



<bound method DataFrame.groupby of         연도   월  분기 청코드 내외항구분 수출입구분명  시설코드       시설명   부두구분명 아외국구분 적공구분  \
0     2024   3   1  신항    외항     수입     6  신항 W 정박지    일반부두   외국선   적컨   
1     2024   4   2  신항    외항     수입     6  신항 W 정박지    일반부두   아국선   적컨   
2     2024  11   4  신항    외항   수입환적     7  신항 U 정박지    일반부두   아국선   공컨   
3     2024  11   4  신항    외항     수입     7  신항 U 정박지    일반부두   아국선   공컨   
4     2024  11   4  신항    외항   수입환적     7  신항 U 정박지    일반부두   아국선   적컨   
...    ...  ..  ..  ..   ...    ...   ...       ...     ...   ...  ...   
2404  2024   7   3  북항    외항   수입환적     8    자성대 부두  컨테이너부두   외국선   적컨   
2405  2024   7   3  북항    외항     수출     8    자성대 부두  컨테이너부두   외국선   적컨   
2406  2024   7   3  북항    외항     수입     8    자성대 부두  컨테이너부두   외국선   적컨   
2407  2024  11   4  북항    외항     수출     8    자성대 부두  컨테이너부두   외국선   적컨   
2408  2024  11   4  북항    외항     수입     8    자성대 부두  컨테이너부두   외국선   적컨   

      컨테이너수(10피트)  컨테이너수(20피트)  컨테이너수(40피트)  컨테이너수(기타)   전체개수     전체물동량   환적

In [28]:
import plotly.express as px

px

<module 'plotly.express' from 'c:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages\\plotly\\express\\__init__.py'>

In [29]:
grp

물동량       개수  TEU_FACTOR
청코드 환적여부                                   
감천  False      2822.00     1516    1.861478
    True       6192.00     3210    1.928972
북항  False   4197997.25  2859718    1.467976
    True    2314512.75  1569611    1.474577
신항  False   6704016.75  4140278    1.619219
    True   11176479.25  6464959    1.728778